# Лабораторная 1. Задания по данным велопарковок SF Bay Area Bike Share

Решение 5 задач с использованием PySpark DataFrame API.

In [3]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.appName("Lab1_Tasks").getOrCreate()

trips = spark.read.csv("/content/trips.csv", header=True, inferSchema=True)
stations = spark.read.csv("/content/stations.csv", header=True, inferSchema=True)

# Убираем строки с пустым duration
trips = trips.filter(F.col("duration").isNotNull())

print("Trips count:", trips.count())
print("Stations count:", stations.count())
trips.printSchema()
stations.printSchema()

Trips count: 115194
Stations count: 70
root
 |-- id: integer (nullable = true)
 |-- duration: integer (nullable = true)
 |-- start_date: string (nullable = true)
 |-- start_station_name: string (nullable = true)
 |-- start_station_id: integer (nullable = true)
 |-- end_date: string (nullable = true)
 |-- end_station_name: string (nullable = true)
 |-- end_station_id: integer (nullable = true)
 |-- bike_id: integer (nullable = true)
 |-- subscription_type: string (nullable = true)
 |-- zip_code: string (nullable = true)

root
 |-- id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- lat: double (nullable = true)
 |-- long: double (nullable = true)
 |-- dock_count: integer (nullable = true)
 |-- city: string (nullable = true)
 |-- installation_date: string (nullable = true)



## Задача 1. Найти велосипед с максимальным временем пробега

In [4]:
bike_total_duration = trips.groupBy("bike_id") \
    .agg(F.sum("duration").alias("total_duration")) \
    .orderBy(F.desc("total_duration"))

max_bike = bike_total_duration.first()
print(f"Велосипед с максимальным временем пробега: bike_id={max_bike['bike_id']}, "
      f"общее время={max_bike['total_duration']} сек ({max_bike['total_duration'] / 3600:.1f} ч)")

bike_total_duration.show(10)

Велосипед с максимальным временем пробега: bike_id=593, общее время=878830 сек (244.1 ч)
+-------+--------------+
|bike_id|total_duration|
+-------+--------------+
|    593|        878830|
|    247|        837790|
|    410|        721972|
|    465|        720288|
|    538|        708680|
|    653|        683482|
|    526|        675520|
|    425|        648550|
|    338|        630230|
|    168|        626668|
+-------+--------------+
only showing top 10 rows


## Задача 2. Найти наибольшее геодезическое расстояние между станциями

In [5]:
# Формула Haversine для расстояния между двумя точками на сфере (в км)
s1 = stations.alias("s1")
s2 = stations.alias("s2")

pairs = s1.crossJoin(s2).filter(F.col("s1.id") < F.col("s2.id"))

# Haversine через PySpark SQL functions
lat1 = F.radians(F.col("s1.lat"))
lat2 = F.radians(F.col("s2.lat"))
dlat = F.radians(F.col("s2.lat") - F.col("s1.lat"))
dlon = F.radians(F.col("s2.long") - F.col("s1.long"))

a = F.sin(dlat / 2) ** 2 + F.cos(lat1) * F.cos(lat2) * F.sin(dlon / 2) ** 2
distance_km = 2 * 6371 * F.asin(F.sqrt(a))

pairs_with_dist = pairs.withColumn("distance_km", distance_km) \
    .select(
        F.col("s1.name").alias("station_1"),
        F.col("s2.name").alias("station_2"),
        "distance_km"
    ) \
    .orderBy(F.desc("distance_km"))

print("Пара станций с наибольшим геодезическим расстоянием:")
pairs_with_dist.show(1, truncate=False)

Пара станций с наибольшим геодезическим расстоянием:
+--------------------------+----------------------+----------------+
|station_1                 |station_2             |distance_km     |
+--------------------------+----------------------+----------------+
|SJSU - San Salvador at 9th|Embarcadero at Sansome|69.9208759542826|
+--------------------------+----------------------+----------------+
only showing top 1 row


## Задача 3. Найти путь велосипеда с максимальным временем пробега через станции

In [6]:
# Берём bike_id из задачи 1
max_bike_id = max_bike['bike_id']
print(f"Велосипед с максимальным пробегом: {max_bike_id}")

# Все поездки этого велосипеда, отсортированные по дате начала
bike_trips = trips.filter(F.col("bike_id") == max_bike_id) \
    .withColumn("start_date_parsed", F.to_timestamp("start_date", "M/d/yyyy H:mm")) \
    .orderBy("start_date_parsed") \
    .select("start_station_name", "end_station_name", "start_date", "end_date", "duration")

print(f"Количество поездок: {bike_trips.count()}")
print("\nПуть через станции (первые 50 поездок):")
bike_trips.show(50, truncate=False)

Велосипед с максимальным пробегом: 593
Количество поездок: 306

Путь через станции (первые 50 поездок):
+---------------------------------------------+---------------------------------------------+---------------+---------------+--------+
|start_station_name                           |end_station_name                             |start_date     |end_date       |duration|
+---------------------------------------------+---------------------------------------------+---------------+---------------+--------+
|Post at Kearney                              |2nd at South Park                            |8/29/2013 12:00|8/29/2013 12:08|430     |
|2nd at South Park                            |2nd at South Park                            |8/29/2013 12:52|8/29/2013 13:08|982     |
|San Francisco Caltrain (Townsend at 4th)     |Civic Center BART (7th at Market)            |8/31/2013 9:07 |8/31/2013 9:25 |1121    |
|South Van Ness at Market                     |Powell Street BART                     

## Задача 4. Найти количество велосипедов в системе

In [7]:
bike_count = trips.select("bike_id").distinct().count()
print(f"Количество велосипедов в системе: {bike_count}")

Количество велосипедов в системе: 689


## Задача 5. Найти пользователей потративших на поездки более 3 часов

In [8]:
# 3 часа = 10800 секунд
long_trips = trips.filter(F.col("duration") > 10800) \
    .select("id", "duration", "start_date", "start_station_name", "end_station_name",
            "bike_id", "subscription_type", "zip_code") \
    .orderBy(F.desc("duration"))

print(f"Количество поездок длительностью более 3 часов: {long_trips.count()}")
print(f"\nРаспределение по типу подписки:")
long_trips.groupBy("subscription_type").count().show()

print("Поездки длительностью более 3 часов (топ-20):")
long_trips.show(20, truncate=False)

Количество поездок длительностью более 3 часов: 1960

Распределение по типу подписки:
+-----------------+-----+
|subscription_type|count|
+-----------------+-----+
|       Subscriber|   96|
|         Customer| 1864|
+-----------------+-----+

Поездки длительностью более 3 часов (топ-20):
+------+--------+----------------+-------------------------------------+----------------------------------------+-------+-----------------+--------+
|id    |duration|start_date      |start_station_name                   |end_station_name                        |bike_id|subscription_type|zip_code|
+------+--------+----------------+-------------------------------------+----------------------------------------+-------+-----------------+--------+
|111309|722236  |11/30/2013 13:29|University and Emerson               |University and Emerson                  |247    |Customer         |94301   |
|129504|619322  |12/18/2013 9:16 |San Jose Diridon Caltrain Station    |SJSU 4th at San Carlos                  |65

In [9]:
spark.stop()